# 18 · Document Ingestion & Parsing

**Idea:** retrieval quality is capped by ingestion quality — garbage text in, garbage chunks out. Before you ever embed anything, raw sources (markdown, HTML, text, PDF) must be parsed to clean prose and normalized so chunking sees consistent input. This is the unglamorous first stage every RAG pipeline needs.

> Runs fully on local files — no LLM, no keys. HTML/PDF parsers import their dependency lazily.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Start with the two most common web formats. `clean_text` normalizes whitespace; `parse_markdown` strips markup; `parse_html` extracts visible text (BeautifulSoup if installed, regex fallback otherwise).

In [ ]:
from rag_lab import clean_text, parse_markdown, parse_html

messy = "# Title\n\n\n\nSome   **bold**   text   with   spaces.\n\n\n\nAnother line."
print('markdown ->', repr(parse_markdown(messy)))

html = "<html><body><h1>RAG</h1><p>Retrieval  augments  generation.</p><script>ignore()</script></body></html>"
print('html     ->', repr(parse_html(html)))

`parse_document` dispatches on file extension, and `ingest_directory` walks a folder into a flat list of chunks tagged with their source filename — exactly the shape the vector store expects.

In [ ]:
from rag_lab import ingest_directory, NOTES_DIR

chunks = ingest_directory(NOTES_DIR, chunk_size=200, overlap=40)
print(f'ingested {len(chunks)} chunks from {NOTES_DIR.name}/')
sources = sorted({c.source for c in chunks})
print('sources:', sources)
print('\nfirst chunk:', chunks[0].source, '->', chunks[0].text[:160], '...')

Those chunks drop straight into a `NumpyStore` — proving ingestion output is retrieval-ready without any extra glue.

In [ ]:
from rag_lab import NumpyStore

store = NumpyStore(provider='ollama'); store.add(chunks)
for h in store.search('how should I choose a chunk size?', k=3):
    print(f'{h.score:.3f}  [{h.chunk.source}]  {h.chunk.text[:90]}...')

**Takeaway:** ingestion turns heterogeneous files into clean, source-tagged chunks — one parser per format, one normalization pass, one uniform output. Everything downstream (chunking, embedding, retrieval) assumes this stage did its job.

Next up → `19_semantic_chunking.ipynb`, where we let *meaning* decide chunk boundaries instead of a fixed word count.